# Inspecting a SUT

In this notebook you will learn how to

- inspect products, industries and final uses
- switch between a formatted table and the numbers behind it
- change how tables are displayed
- save an inspection to Excel

## Load the data

See [01_getting_started](01_getting_started.ipynb) for an explanation of these steps.

In [ ]:
import sys
from pathlib import Path

# Location of sutlab. Absolute, so that the notebook can be copied to another folder
sutlab_folder = Path(r"C:\Users\DstMove\Desktop\claude\projects\sutlab")
sys.path.insert(0, str(sutlab_folder))

import sutlab

data_folder = sutlab_folder / "docs" / "examples" / "data"
years = [2016, 2017, 2018, 2019]

metadata = sutlab.load_metadata_from_excel(
    data_folder / "ta_columns.xlsx",
    data_folder / "ta_classifications.xlsx",
)

current_paths = [data_folder / f"ta_l_{year}.parquet" for year in years]
sut = sutlab.load_sut_from_separated_parquet(years, current_paths, metadata, "current_year")

## Products

`inspect_products` collects a number of tables about one or more products. We use wine as the example.

In [ ]:
products = sut.inspect_products("V220400")

`tables_description` lists the tables in the inspection.

In [ ]:
products.tables_description

The `balance` table shows supply at the top and use at the bottom. Use is in purchasers' prices, so the price layers are added to supply to make the two sides comparable.

In [ ]:
products.balance

The `use` table splits each transaction into industries and consumption groups.

In [ ]:
products.use

Most tables also exist with the ending `_distribution` (shares) and `_growth` (change from the year before).

In [ ]:
products.balance_growth

Price layers can be shown as rates. Each layer is divided by the value it is added on top of. For example, VAT is divided by the basic price plus margins and other taxes.

In [ ]:
products.price_layers_rates

## Formatted table or numbers

`products.balance` is formatted for reading. `products.data.balance` is an ordinary DataFrame, which you can calculate with.

In [ ]:
products.data.balance

## Industries

`inspect_industries` shows output, input and gross value added. Here it is agriculture and horticulture.

In [ ]:
industries = sut.inspect_industries("010000")
industries.balance

An industry uses many products. The summary table shows how concentrated the input is: how many products there are, how many of them it takes to cover a given share of the total, and the size of the typical product.

In [ ]:
industries.use_products_summary

## Final uses

`inspect_final_uses` covers consumption, investment, inventories and exports. Here it is household consumption of wine and spirits and of beer.

In [ ]:
final_uses = sut.inspect_final_uses("3110", categories=["02112", "02130"])
final_uses.use_categories

In [ ]:
final_uses.use_products

## Change the display

Functions that start with `set_display_` change how the tables look. They do not change the numbers in `.data`, and they can be put after each other.

The data is in thousand DKK, so dividing by 1,000 shows million DKK.

In [ ]:
final_uses_million = final_uses.set_display_unit(1000).set_display_decimals(0)
final_uses_million.use_products

Long tables can be cut down to the largest rows. Here are the ten products that agriculture and horticulture used the most in 2019.

In [ ]:
industries_short = industries.set_display_values_n_largest(10, 2019)
industries_short = industries_short.set_display_sort_column(2019)
industries_short.use_products

## Save to Excel

`write_to_excel` writes each table to its own sheet, with the formatting kept. Use `tables` to choose which ones. If you leave it out, you get them all.

In [ ]:
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

products.write_to_excel(
    output_folder / "wine.xlsx",
    tables=["balance", "use", "price_layers_rates"],
)

---
Next: [03_gdp_and_volumes](03_gdp_and_volumes.ipynb)